In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import relativedelta
from ChildProject.projects import ChildProject
from ChildProject.annotations import AnnotationManager

DATA_PATH = Path('/home/engaclew/neurogen')

# Read measures
aclew_measures = pd.read_csv(DATA_PATH / 'aclew_measures_chunks.csv').fillna(0)
lena_measures = pd.read_csv(DATA_PATH / 'lena_measures_chunks.csv').fillna(0)
human_measures = pd.read_csv(DATA_PATH / 'human_measures_chunks.csv').fillna(0)

# Read metadata
children = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/children.csv')
recordings = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'date_iso', 'recording_filename', 'child_sex', 'child_dob']]
def diff_month(row):
    d1 = datetime.strptime(row['date_iso'], '%Y-%m-%d')
    d2 = datetime.strptime(row['child_dob'], '%Y-%m-%d')
    return (d1.year - d2.year) * 12 + d1.month - d2.month
recordings_data['age'] = recordings_data.apply(lambda row: diff_month(row), axis=1)


aclew_measures = aclew_measures.merge(recordings_data, how='left', on='recording_filename')
lena_measures = lena_measures.merge(recordings_data, how='left', on='recording_filename')
human_measures = human_measures.merge(recordings_data, how='left', on='recording_filename')

def compute_CVC(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    return data

aclew_measures = compute_CVC(aclew_measures)
human_measures = compute_CVC(human_measures)

# Somehow 5s_CTC is not accepted by statsmodels for the linear mixed model... (doesn't like variables that start with numbers)
aclew_measures = aclew_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})
lena_measures = lena_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})
human_measures = human_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})

print(human_measures.groupby('group_id').size())
print(f"Total = {human_measures.groupby('group_id').size().sum()}")


               recording_filename  segment_onset  segment_offset  child_id  \
0      20180530_181655_022873.wav        1639000         1759000      3321   
1      20180530_181655_022873.wav        2072000         2192000      3321   
2      20180530_181655_022873.wav        2202000         2322000      3321   
3      20180530_181655_022873.wav        5817000         5937000      3321   
4      20180530_181655_022873.wav        6620000         6740000      3321   
..                            ...            ...             ...       ...   
745  20231025_115855_045737_2.wav       16289000        16409000      6731   
746  20231025_115855_045737_2.wav       16567000        16687000      6731   
747  20231025_115855_045737_2.wav       18264000        18384000      6731   
748  20231025_115855_045737_2.wav       28611000        28731000      6731   
749  20231025_115855_045737_2.wav       36581000        36701000      6731   

     duration_alice  wc_fem  wc_mal  wc_adu  duration_vcm  non_

In [2]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

def print_full_mixed_effects_results(human_measures, automatic_measures, dvs):
    """
    Print mixed effects results table for multiple dependent variables.
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    data = pd.merge(
        human_measures, 
        automatic_measures,
        on=['recording_filename', 'segment_onset', 'segment_offset', 'child_id', 'group_id', 'child_sex', 'age'],
        suffixes=('_estimated', '_true')
    ) 

    data['group_id'] = data['group_id'].replace({
        'low_risk': '_low_risk',
        'angelman_syndrome': 'angelman_syndrome',
        'autism_sibling': 'autism_sibling',
        'down_syndrome': 'down_syndrome',
        'fragile_x_syndrome': 'fragile_x_syndrome'
    })
    print("\nTable X. Mixed Effects Analysis Results")
    print("-" * 100)
    print(f"{'Dependent Variable':<20} {'Predictor':<20} {'β':>8} {'SE':>8} {'z':>8} {'p':>10} {'η²p':>8}")
    print("-" * 100)
    
    for dv in dvs:
        # Fit model
        model = smf.mixedlm(
            f"{dv}_true ~ {dv}_estimated + group_id + age + child_sex", 
            data=data,
            groups="child_id"
        ).fit()
        
        # Get results table
        results = model.summary().tables[1]
        
        # Process results (skip the last row which is for random effects)
        for idx in range(len(results)-1):
            name = results.index[idx]
            beta = float(results.iloc[idx, 0])  # Coefficient (beta)
            se = float(results.iloc[idx, 1])    # Standard error
            z_stat = float(results.iloc[idx, 2])
            p_val = float(results.iloc[idx, 3])
            
            # Clean up predictor names
            clean_name = (name.replace('group_id[T.', '')
                            .replace('child_sex[T.', '')
                            .replace(']', '')
                            .replace('_', ' '))
            
            # Calculate partial eta-squared
            df_resid = model.df_resid
            eta_sq = (z_stat**2) / (z_stat**2 + df_resid)
            
            # Format p-value
            if p_val < 0.001:
                p_value = "< .001***"
            else:
                p_value = f"{p_val:.3f}"
                if p_val < 0.01:
                    p_value += "**"
                elif p_val < 0.05:
                    p_value += "*"
            
            print(f"{dv:<20} {clean_name:<20} {beta:>8.2f} {se:>8.2f} {z_stat:>8.2f} {p_value:>10}")
        
        print()
    
    print("-" * 80)
    print("Note: η²p = partial eta-squared")
    print("* p < .05, ** p < .01, *** p < .001")

# Example usage:
dvs = ['CTC', 'CVC', 'AWC']
print('LENA')
print_full_mixed_effects_results(human_measures, lena_measures, dvs)
print('ACLEW')
print_full_mixed_effects_results(human_measures, aclew_measures, dvs)

LENA

Table X. Mixed Effects Analysis Results
----------------------------------------------------------------------------------------------------
Dependent Variable   Predictor                   β       SE        z          p      η²p
----------------------------------------------------------------------------------------------------
CTC                  Intercept                0.36     0.50     0.73      0.467
CTC                  angelman syndrome        0.42     0.29     1.42      0.155
CTC                  autism sibling           0.27     0.30     0.90      0.367
CTC                  down syndrome            0.46     0.29     1.58      0.115
CTC                  fragile x syndrome       0.46     0.29     1.56      0.118
CTC                  m                       -0.04     0.19    -0.23      0.815
CTC                  CTC estimated            0.16     0.01    25.12  < .001***
CTC                  age                     -0.01     0.02    -0.67      0.500

CVC                  I

In [3]:
print("Low risk:")
print(f"Human AWC mean: {human_measures[human_measures['group_id']=='low_risk']['AWC'].mean()}")
print(f"LENA AWC mean: {lena_measures[lena_measures['group_id']=='low_risk']['AWC'].mean()}")
print(f"ACLEW AWC mean: {aclew_measures[aclew_measures['group_id']=='low_risk']['AWC'].mean()}")

print("\nAngelman:")
print(f"Human AWC mean: {human_measures[human_measures['group_id']=='angelman_syndrome']['AWC'].mean()}")
print(f"LENA AWC mean: {lena_measures[lena_measures['group_id']=='angelman_syndrome']['AWC'].mean()}")
print(f"ACLEW AWC mean: {aclew_measures[aclew_measures['group_id']=='angelman_syndrome']['AWC'].mean()}")

Low risk:
Human AWC mean: 48.12
LENA AWC mean: 22.5378
ACLEW AWC mean: 64.0182

Angelman:
Human AWC mean: 57.78666666666667
LENA AWC mean: 44.31286666666667
ACLEW AWC mean: 92.71993333333333


In [8]:
def compare_models_with_anova(human_measures, automatic_measures, dvs):
    """
    Compare models with and without diagnostic group using likelihood ratio test.
    Reports additional variance explained by diagnostic group.
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    
    data = pd.merge(
        human_measures, 
        automatic_measures,
        on=['recording_filename', 'segment_onset', 'segment_offset', 'child_id', 'group_id', 'child_sex', 'age'],
        suffixes=('_estimated', '_true')
    ) 
    
    data['group_id'] = data['group_id'].replace({
        'low_risk': '_low_risk',
        'angelman_syndrome': 'angelman_syndrome',
        'autism_sibling': 'autism_sibling',
        'down_syndrome': 'down_syndrome',
        'fragile_x_syndrome': 'fragile_x_syndrome'
    })
    
    print("\nModel Comparison: With vs. Without Diagnostic Group")
    print("-" * 150)
    print(f"{'DV':<10} {'Model':<35} {'LogLik':>12} {'AIC':>12} {'R²m':>8} {'R²auto':>10} {'χ²':>10} {'df':>5} {'p-value':>12} {'ΔR²':>10}")
    print("-" * 150)
    
    for dv in dvs:
        # Model with automatic count ONLY (for R² comparison)
        model_auto_only = smf.mixedlm(
            f"{dv}_true ~ {dv}_estimated", 
            data=data,
            groups="child_id"
        ).fit()
        
        # Calculate R² for automatic count only
        var_fixed_auto = np.var(model_auto_only.fittedvalues)
        var_random_auto = float(model_auto_only.cov_re.iloc[0, 0])
        var_residual_auto = model_auto_only.scale
        total_var_auto = var_fixed_auto + var_random_auto + var_residual_auto
        r2_auto_only = var_fixed_auto / total_var_auto
        
        # Model without diagnostic group - USE ML NOT REML
        model_base = smf.mixedlm(
            f"{dv}_true ~ {dv}_estimated + age + child_sex", 
            data=data,
            groups="child_id"
        ).fit(reml=False)
        
        # Model with diagnostic group - USE ML NOT REML
        model_full = smf.mixedlm(
            f"{dv}_true ~ {dv}_estimated + group_id + age + child_sex", 
            data=data,
            groups="child_id"
        ).fit(reml=False)
        
        # Calculate marginal R² for base model
        var_fixed_base = np.var(model_base.fittedvalues)
        var_random_base = float(model_base.cov_re.iloc[0, 0])
        var_residual_base = model_base.scale
        total_var_base = var_fixed_base + var_random_base + var_residual_base
        r2_marginal_base = var_fixed_base / total_var_base
        
        # Calculate marginal R² for full model
        var_fixed_full = np.var(model_full.fittedvalues)
        var_random_full = float(model_full.cov_re.iloc[0, 0])
        var_residual_full = model_full.scale
        total_var_full = var_fixed_full + var_random_full + var_residual_full
        r2_marginal_full = var_fixed_full / total_var_full
        
        # Calculate difference in R²
        delta_r2 = r2_marginal_full - r2_marginal_base
        
        # Likelihood ratio test
        lr_stat = 2 * (model_full.llf - model_base.llf)
        df_diff = len(model_full.params) - len(model_base.params)
        from scipy import stats
        p_value = stats.chi2.sf(lr_stat, df_diff)
        
        # Format p-value
        if p_value < 0.001:
            p_str = "< .001***"
        else:
            p_str = f"{p_value:.3f}"
            if p_value < 0.01:
                p_str += "**"
            elif p_value < 0.05:
                p_str += "*"
        
        # Print results
        print(f"{dv:<10} {'Auto count only':<35} {'':<12} {'':<12} {'':<8} {r2_auto_only:>10.3f}")
        print(f"{dv:<10} {'Base (no group)':<35} {model_base.llf:>12.2f} {model_base.aic:>12.2f} {r2_marginal_base:>8.3f}")
        print(f"{dv:<10} {'Full (with group)':<35} {model_full.llf:>12.2f} {model_full.aic:>12.2f} {r2_marginal_full:>8.3f} {'':<10} {lr_stat:>10.2f} {df_diff:>5} {p_str:>12} {delta_r2:>10.3f}")
        print()
    
    print("-" * 150)
    print("Note: R²m = Marginal R² (variance explained by fixed effects)")
    print("      R²auto = Variance explained by automatic count alone")
    print("      ΔR² = Additional variance explained by adding diagnostic group")
    print("      χ² = likelihood ratio statistic; Models fitted with ML for comparison")
    print("* p < .05, ** p < .01, *** p < .001")

# Run the comparison
print('\n=== LENA ===')
compare_models_with_anova(human_measures, lena_measures, dvs)

print('\n=== ACLEW ===')
compare_models_with_anova(human_measures, aclew_measures, dvs)


=== LENA ===

Model Comparison: With vs. Without Diagnostic Group
------------------------------------------------------------------------------------------------------------------------------------------------------
DV         Model                                     LogLik          AIC      R²m     R²auto         χ²    df      p-value        ΔR²
------------------------------------------------------------------------------------------------------------------------------------------------------
CTC        Auto count only                                                             0.506
CTC        Base (no group)                         -1411.60      2835.19    0.506
CTC        Full (with group)                       -1409.45      2838.91    0.510                  4.29     4        0.369      0.004

CVC        Auto count only                                                             0.559
CVC        Base (no group)                         -2149.71      4311.41    0.560
CVC        F

In [3]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

def add_significance_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

param_name = {
    'group_id[T.angelman_syndrome]': 'Angelman',
    'group_id[T.autism_sibling]': 'Autism sib.',
    'group_id[T.down_syndrome]': 'Down',
    'group_id[T.fragile_x_syndrome]': 'Fragile X',
    'CVC_estimated': 'CVC_estimated',
    'CTC_5s': 'CTC_estimated',
    'wc_adu_estimated': 'AWC_estimated',
    'age': 'age',
    'child_id': 'child_id'

}

def biases_mixed_model(hyp, ref, measure):
    merged_data = pd.merge(
        hyp, 
        ref, 
        on=['recording_filename', 'segment_onset', 'segment_offset', 'child_id', 'group_id', 'age'],
        suffixes=('_estimated', '_true')
    ) 
    
    # Merge the data
    merged_data['group_id'] = merged_data['group_id'].replace({
        'low_risk': '_low_risk',
        'angelman_syndrome': 'angelman_syndrome',
        'autism_sibling': 'autism_sibling',
        'down_syndrome': 'down_syndrome',
        'fragile_x_syndrome': 'fragile_x_syndrome'
    })

    # Fit the mixed-effects model
    # Note: Changed the formula to match your variables but kept true as dependent variable
    model = smf.mixedlm(
        f"{measure}_true ~ {measure}_estimated + group_id + age", 
        data=merged_data,
        groups="child_id"
    )

    result = model.fit()
    conf_int = result.conf_int()
    #print(result.summary())
    print("\nFixed Effects with 95% Confidence Intervals:")
    print("Parameter Estimate CI_lower CI_upper p-value Significance")
    for param in result.params.index:
        # Get significance stars
        p_value = result.pvalues[param]
        stars = ''
        if p_value < 0.001:
            stars = '***'
        elif p_value < 0.01:
            stars = '**'
        elif p_value < 0.05:
            stars = '*'

        param_label = param
        if param in param_name:
            param_label = param_name[param]
        print(f"{param_label} {result.params[param]:.2f} {conf_int.loc[param][0]:.2f} {conf_int.loc[param][1]:.2f} {p_value:.3f} {stars}")

print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='CVC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='CVC')

ACLEW

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept -2.69 -6.20 0.83 0.134 
Angelman -0.43 -2.49 1.63 0.683 
Autism sib. 1.21 -0.83 3.26 0.245 
Down -0.89 -2.95 1.17 0.397 
Fragile X -0.34 -2.39 1.72 0.748 
CVC_estimated 1.30 1.21 1.39 0.000 ***
age 0.19 0.03 0.34 0.018 *
child_id Var 0.05 -0.00 0.09 0.068 


LENA

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept 2.03 -2.26 6.33 0.353 
Angelman -2.08 -4.61 0.44 0.106 
Autism sib. -0.25 -2.77 2.26 0.843 
Down 0.29 -2.25 2.82 0.825 
Fragile X -0.19 -2.71 2.34 0.885 
CVC_estimated 1.22 1.14 1.31 0.000 ***
age 0.11 -0.09 0.30 0.278 
child_id Var 0.10 0.03 0.18 0.006 **


In [4]:
print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='AWC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='AWC')

ACLEW

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept 19.46 -5.14 44.07 0.121 
Angelman -7.01 -21.50 7.48 0.343 
Autism sib. -10.79 -25.21 3.63 0.142 
Down 1.92 -12.59 16.42 0.796 
Fragile X -2.10 -16.54 12.35 0.776 
AWC_estimated 0.60 0.57 0.64 0.000 ***
age -0.49 -1.58 0.61 0.384 
child_id Var 0.08 0.02 0.14 0.014 *


LENA

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept 24.92 2.75 47.09 0.028 *
Angelman -10.49 -23.58 2.60 0.116 
Autism sib. -13.68 -26.70 -0.66 0.039 *
Down -2.38 -15.49 10.72 0.721 
Fragile X -16.80 -29.88 -3.72 0.012 *
AWC_estimated 0.92 0.86 0.98 0.000 ***
age 0.12 -0.86 1.11 0.807 
child_id Var 0.05 -0.00 0.10 0.056 


In [5]:
print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='CTC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='CTC')

ACLEW

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept -2.69 -5.07 -0.31 0.026 *
Angelman 0.32 -1.08 1.72 0.655 
Autism sib. 0.14 -1.25 1.53 0.841 
Down -1.68 -3.08 -0.28 0.019 *
Fragile X -0.43 -1.82 0.97 0.548 
CTC_estimated 0.99 0.94 1.04 0.000 ***
age 0.12 0.02 0.23 0.022 *
child_id Var 0.03 -0.01 0.07 0.171 


LENA

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept -0.89 -4.43 2.64 0.620 
Angelman -1.53 -3.61 0.56 0.151 
Autism sib. -0.79 -2.86 1.29 0.458 
Down -1.88 -3.96 0.21 0.078 
Fragile X -1.35 -3.43 0.73 0.204 
CTC_estimated 2.91 2.68 3.13 0.000 ***
age 0.20 0.04 0.36 0.012 *
child_id Var 0.06 0.01 0.12 0.029 *


In [6]:
# Running analyses, removing clips without child speech
mask = (human_measures.voc_dur_chi != 0) & (human_measures.AWC != 0)
human_measures = human_measures[mask]
lena_measures = lena_measures[mask]
aclew_measures = aclew_measures[mask]

print(human_measures.groupby('group_id').size())
print(f"Total = {human_measures.groupby('group_id').size().sum()}")

print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='CVC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='CVC')

print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='AWC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='AWC')

print("ACLEW")
biases_mixed_model(aclew_measures, human_measures, measure='CTC')
print("\n\nLENA")
biases_mixed_model(lena_measures, human_measures, measure='CTC')

group_id
angelman_syndrome      93
autism_sibling         97
down_syndrome         110
fragile_x_syndrome     88
low_risk               82
dtype: int64
Total = 470
ACLEW

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept -3.86 -7.99 0.28 0.068 
Angelman -0.44 -2.90 2.01 0.723 
Autism sib. 1.64 -0.76 4.04 0.179 
Down -1.43 -3.79 0.92 0.233 
Fragile X 0.22 -2.23 2.67 0.861 
CVC_estimated 1.25 1.13 1.38 0.000 ***
age 0.31 0.14 0.48 0.000 ***
child_id Var 0.02 -0.03 0.07 0.464 


LENA

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lower CI_upper p-value Significance
Intercept 3.43 -1.81 8.67 0.200 
Angelman -3.82 -6.95 -0.68 0.017 *
Autism sib. -1.04 -4.15 2.06 0.510 
Down -1.43 -4.52 1.66 0.365 
Fragile X -0.62 -3.80 2.56 0.703 
CVC_estimated 1.06 0.96 1.16 0.000 ***
age 0.22 -0.01 0.45 0.058 
child_id Var 0.13 0.01 0.25 0.028 *
ACLEW

Fixed Effects with 95% Confidence Intervals:
Parameter Estimate CI_lowe

In [20]:
aclew_measures

,recording_filename,segment_onset,segment_offset,child_id_x,duration_alice,wc_fem,wc_mal,AWC,duration_vcm,non_can_voc_CHI,...,CTC,speechlike_pitch,nonspeechlike_pitch,child_id_y,group_id,date_iso,child_sex,child_dob,age,CVC
0,20180530_181655_022873.wav,1639000,1759000,3321,120000,58.72,0.88,59.60,120000,5,...,30,"[228.48704510485052, 486.9420404296455, 394.43...","[284.12270707020286, 403.0400608703906, 429.70...",3321,low_risk,2018-05-21,f,2016-08-21,21,5
1,20180530_181655_022873.wav,2072000,2192000,3321,120000,40.38,0.00,40.38,120000,16,...,9,"[518.8369582195888, 409.8143921244038, 283.117...",[372.2728234468482],3321,low_risk,2018-05-21,f,2016-08-21,21,17
2,20180530_181655_022873.wav,2202000,2322000,3321,120000,46.32,7.72,54.04,120000,10,...,9,"[nan, nan, 326.60608559543766, 397.34887006092...",[456.42752176149645],3321,low_risk,2018-05-21,f,2016-08-21,21,10
3,20180530_181655_022873.wav,5817000,5937000,3321,120000,33.58,0.00,33.58,120000,8,...,4,"[471.73529298938456, 455.77709433980056, 351.3...",[],3321,low_risk,2018-05-21,f,2016-08-21,21,8
4,20180530_181655_022873.wav,6620000,6740000,3321,120000,61.70,0.00,61.70,120000,11,...,4,"[474.5115338430446, 323.3997502281562, 300.831...","[567.1308660183602, 510.45102362133207, 384.10...",3321,low_risk,2018-05-21,f,2016-08-21,21,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,20231025_115855_045737_2.wav,16289000,16409000,6731,120000,126.16,43.31,169.47,120000,15,...,30,"[364.9781975528883, 294.2447392710183, 348.063...",[504.7319520767478],6731,low_risk,2023-10-20,m,2021-10-22,24,18
746,20231025_115855_045737_2.wav,16567000,16687000,6731,120000,25.11,4.85,29.96,120000,7,...,13,"[408.4288605221903, 382.6535837739436, 275.534...","[429.43839439959316, 435.55151519713627, 355.3...",6731,low_risk,2023-10-20,m,2021-10-22,24,8
747,20231025_115855_045737_2.wav,18264000,18384000,6731,120000,0.00,0.00,0.00,120000,0,...,0,[],[],6731,low_risk,2023-10-20,m,2021-10-22,24,0
748,20231025_115855_045737_2.wav,28611000,28731000,6731,120000,37.17,30.55,67.72,120000,8,...,13,"[356.64924857986443, 555.4325081117244, 435.47...",[376.9298446973081],6731,low_risk,2023-10-20,m,2021-10-22,24,9


In [23]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import relativedelta
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

DATA_PATH = Path('/home/engaclew/neurogen')

# Read measures
aclew_measures = pd.read_csv(DATA_PATH / 'aclew_measures_chunks.csv').fillna(0)
lena_measures = pd.read_csv(DATA_PATH / 'lena_measures_chunks.csv').fillna(0)
human_measures = pd.read_csv(DATA_PATH / 'human_measures_chunks.csv').fillna(0)

# Read metadata
children = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/children.csv')
recordings = pd.read_csv(DATA_PATH / 'data/L3_HIPAA_LENA_cleaned/metadata/recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'date_iso', 'recording_filename', 'child_sex', 'child_dob']]
def diff_month(row):
    d1 = datetime.strptime(row['date_iso'], '%Y-%m-%d')
    d2 = datetime.strptime(row['child_dob'], '%Y-%m-%d')
    return (d1.year - d2.year) * 12 + d1.month - d2.month
recordings_data['age'] = recordings_data.apply(lambda row: diff_month(row), axis=1)


aclew_measures = aclew_measures.merge(recordings_data, how='left', on='recording_filename')
lena_measures = lena_measures.merge(recordings_data, how='left', on='recording_filename')
human_measures = human_measures.merge(recordings_data, how='left', on='recording_filename')
print(aclew_measures)
def compute_CVC(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    return data

aclew_measures = compute_CVC(aclew_measures)
human_measures = compute_CVC(human_measures)

aclew_measures = aclew_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})
lena_measures = lena_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})
human_measures = human_measures.rename(columns={'5s_CTC': 'CTC', 'wc_adu': 'AWC'})
# Compute absolute percentage errors
def compute_error(x, y):
    return x - y 

# Create combined dataframes with errors
lena_data = pd.DataFrame()
aclew_data = pd.DataFrame()

for col in ['CTC', 'AWC', 'CVC', 'child_id', 'group_id', 'child_sex', 'age']:
    lena_data[col] = human_measures[col]
    aclew_data[col] = human_measures[col]

for measure in ['CTC', 'AWC', 'CVC']:
    lena_data[f'{measure}_mape'] = compute_error(human_measures[measure].values, lena_measures[measure].values)
    aclew_data[f'{measure}_mape'] = compute_error(human_measures[measure].values, aclew_measures[measure].values)



               recording_filename  segment_onset  segment_offset  child_id  \
0      20180530_181655_022873.wav        1639000         1759000      3321   
1      20180530_181655_022873.wav        2072000         2192000      3321   
2      20180530_181655_022873.wav        2202000         2322000      3321   
3      20180530_181655_022873.wav        5817000         5937000      3321   
4      20180530_181655_022873.wav        6620000         6740000      3321   
..                            ...            ...             ...       ...   
745  20231025_115855_045737_2.wav       16289000        16409000      6731   
746  20231025_115855_045737_2.wav       16567000        16687000      6731   
747  20231025_115855_045737_2.wav       18264000        18384000      6731   
748  20231025_115855_045737_2.wav       28611000        28731000      6731   
749  20231025_115855_045737_2.wav       36581000        36701000      6731   

     duration_alice  wc_fem  wc_mal  wc_adu  duration_vcm  non_

In [25]:
# Recode group_id for statsmodels
lena_data['group_id'] = lena_data['group_id'].replace({
    'low_risk': '_low_risk',
    'angelman_syndrome': 'angelman_syndrome',
    'autism_sibling': 'autism_sibling',
    'down_syndrome': 'down_syndrome',
    'fragile_x_syndrome': 'fragile_x_syndrome'
})

aclew_data['group_id'] = aclew_data['group_id'].replace({
    'low_risk': '_low_risk',
    'angelman_syndrome': 'angelman_syndrome',
    'autism_sibling': 'autism_sibling',
    'down_syndrome': 'down_syndrome',
    'fragile_x_syndrome': 'fragile_x_syndrome'
})

# Test for group differences in absolute errors
def test_error_differences(data, algorithm_name):
    """
    Test whether errors differ across diagnostic groups
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    
    measures = ['AWC', 'CTC', 'CVC']
    
    print(f"\n{'='*80}")
    print(f"{algorithm_name} - Error by Group")
    print(f"{'='*80}\n")
    
    for measure in measures:
        print(f"\n=== {measure} ===")
        
        try:
            model_data = data[[f'{measure}_mape', 'group_id', 'child_sex', 'age', 'child_id']].dropna()
            
            # Remove infinite values
            model_data = model_data[~np.isinf(model_data[f'{measure}_mape'])]
            
            if len(model_data) == 0:
                print("No valid data")
                continue
            
            model = smf.mixedlm(
                f"{measure}_mape ~ group_id + age + child_sex",
                data=model_data,
                groups="child_id"
            ).fit()
            
            print(model.summary().tables[1])
            
        except Exception as e:
            print(f"Error: {e}")

# Run analysis
print("\n" + "="*80)
print("TESTING: Do algorithms perform differently across diagnostic groups?")
print("="*80)
test_error_differences(lena_data, "LENA®")
test_error_differences(aclew_data, "ACLEW")


TESTING: Do algorithms perform differently across diagnostic groups?

LENA® - Error by Group


=== AWC ===
                                  Coef. Std.Err.       z  P>|z|   [0.025  \
Intercept                        23.555   11.842   1.989  0.047    0.346   
group_id[T.angelman_syndrome]   -12.357    6.992  -1.767  0.077  -26.061   
group_id[T.autism_sibling]      -15.421    7.028  -2.194  0.028  -29.197   
group_id[T.down_syndrome]        -4.135    6.931  -0.597  0.551  -17.720   
group_id[T.fragile_x_syndrome]  -18.539    6.925  -2.677  0.007  -32.111   
child_sex[T.m]                   -0.505    4.597  -0.110  0.912   -9.515   
age                               0.114    0.526   0.216  0.829   -0.918   
child_id Var                    109.425    1.204                           

                                0.975]  
Intercept                       46.764  
group_id[T.angelman_syndrome]    1.347  
group_id[T.autism_sibling]      -1.646  
group_id[T.down_syndrome]        9.449  
gr